In [ ]:
!pip install timm albumentations

In [ ]:
import os, cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
IMG_DIR1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1/"
IMG_DIR2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2/"
CSV_PATH = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"

In [ ]:
df = pd.read_csv(CSV_PATH)

df['path'] = df['image_id'].apply(lambda x:
    os.path.join(IMG_DIR1, x+'.jpg') if os.path.exists(os.path.join(IMG_DIR1, x+'.jpg'))
    else os.path.join(IMG_DIR2, x+'.jpg')
)

le = LabelEncoder()
df['label'] = le.fit_transform(df['dx'])

NUM_CLASSES = df['label'].nunique()

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df['label']),
    y=df['label']
)

class_weights = torch.tensor(class_weights, dtype=torch.float).cuda()

In [ ]:
train_aug = A.Compose([
    A.Resize(224, 224),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),

    A.Affine(
        scale=(0.9, 1.1),
        translate_percent=(0.05, 0.05),
        rotate=(-30, 30),
        shear=(-10, 10),
        p=0.7
    ),

    A.OneOf([
        A.MotionBlur(blur_limit=5),
        A.MedianBlur(blur_limit=5),
        A.GaussianBlur(blur_limit=5),
    ], p=0.3),

    A.RandomBrightnessContrast(p=0.5),

    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=15,
        val_shift_limit=10,
        p=0.5
    ),

    A.CoarseDropout(
        num_holes_range=(1, 8),
        hole_height_range=(0.05, 0.2),
        hole_width_range=(0.05, 0.2),
        fill=0,
        p=0.5
    ),

    A.Normalize(),
    ToTensorV2()
])

valid_aug = A.Compose([
    A.Resize(224, 224),
    A.Normalize(),
    ToTensorV2()
])

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, df, aug):
        self.df = df.reset_index(drop=True)
        self.aug = aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = self.aug(image=img)['image']
        label = torch.tensor(row['label'], dtype=torch.long)

        return img, label

In [ ]:
class Model(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.model = timm.create_model(backbone, pretrained=True, num_classes=NUM_CLASSES)

    def forward(self, x):
        return self.model(x)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).cuda()

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

In [ ]:
def tta(model, img):
    imgs = [
        img,
        torch.flip(img, [2]),
        torch.flip(img, [3])
    ]

    preds = []
    for im in imgs:
        preds.append(torch.softmax(model(im), dim=1))

    return torch.mean(torch.stack(preds), dim=0)

Model training function variation for model testing

In [ ]:
from torch.amp import autocast, GradScaler
def train_fold(df, train_idx, val_idx, backbone, fold):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ✅ REMOVED Weighted Sampling completely

    train_loader = DataLoader(
        SkinDataset(df.iloc[train_idx], train_aug),
        batch_size=12,              # (small boost, safe change)
        shuffle=True,              # ✅ important
        num_workers=0
    )

    val_loader = DataLoader(
        SkinDataset(df.iloc[val_idx], valid_aug),
        batch_size=12,
        shuffle=False,
        num_workers=0
    )

    model = Model(backbone).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2
    )

    scaler = GradScaler()

    best_acc = 0

    for epoch in range(25):

        model.train()

        # Freeze logic (same as yours)
        if epoch < 2:
            for name, p in model.named_parameters():
                if "head" in name or "fc" in name:
                    p.requires_grad = True
                else:
                    p.requires_grad = False
        else:
            for p in model.parameters():
                p.requires_grad = True

        for img, label in train_loader:
        img, label = img.to(device), label.to(device)
        img, y_a, y_b, lam = mixup(img, label)
    
        optimizer.zero_grad()
    
        with autocast(device_type='cuda'):   # <-- specify device_type
            pred = model(img)
            loss = lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)
        
        scaler.scale(loss).backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        scaler.step(optimizer)
        scaler.update()

            # ✅ CHANGE 2: Gradient Clipping (ADDED)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

        scheduler.step()

        # VALIDATION WITH TTA
        model.eval()
        preds, labels_list = [], []

        with torch.no_grad():
            for img, label in val_loader:
                img = img.to(device)

                out = tta(model, img)
                p = torch.argmax(out, dim=1)

                preds.extend(p.cpu().numpy())
                labels_list.extend(label.numpy())

        acc = accuracy_score(labels_list, preds)
        print(f"{backbone} Fold {fold} Epoch {epoch} Acc {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f"{backbone}_fold{fold}.pth")

In [ ]:
from torch.amp import autocast, GradScaler  # unified import for PyTorch >=2.1

def train_fold(df, train_idx, val_idx, backbone, fold):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(
        SkinDataset(df.iloc[train_idx], train_aug),
        batch_size=10,
        shuffle=True,
        num_workers=4
    )

    val_loader = DataLoader(
        SkinDataset(df.iloc[val_idx], valid_aug),
        batch_size=10,
        shuffle=False,
        num_workers=4
    )

    model = Model(backbone).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    scaler = GradScaler()

    best_acc = 0

    for epoch in range(10):

        model.train()

        # Freeze logic
        if epoch < 2:
            for name, p in model.named_parameters():
                if "head" in name or "fc" in name:
                    p.requires_grad = True
                else:
                    p.requires_grad = False
        else:
            for p in model.parameters():
                p.requires_grad = True

        for img, label in train_loader:
            img, label = img.to(device), label.to(device)
            img, y_a, y_b, lam = mixup(img, label)

            optimizer.zero_grad()

            # ✅ Correct autocast usage
            with autocast(device_type = "cuda"):  # unified amp; no device_type needed
                pred = model(img)
                loss = lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

            scaler.scale(loss).backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

        scheduler.step()

        # VALIDATION
        model.eval()
        preds, labels_list = [], []

        with torch.no_grad():
            for img, label in val_loader:
                img = img.to(device)
                out = tta(model, img)
                p = torch.argmax(out, dim=1)
                preds.extend(p.cpu().numpy())
                labels_list.extend(label.numpy())

        acc = accuracy_score(labels_list, preds)
        print(f"{backbone} Fold {fold} Epoch {epoch} Acc {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f"{backbone}_fold{fold}.pth")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

backbones = [
    'tf_efficientnet_b4_ns',
    'convnext_base'
]

for backbone in backbones:
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):

        print(f"\n Training {backbone} Fold {fold}")

        try:
            train_fold(df, train_idx, val_idx, backbone, fold)
        except Exception as e:
            import traceback
            print("❌ ERROR:")
            traceback.print_exc()